# MEGC walrasiano de 3 sectores: shock del datacenter Stargate Argentina.


Economía pequeña y abierta con tres sectores:

* Energía (E): generación nacional, no transable, usa trabajo y capital.
* Datacenter (D): Leontief en energía, trabajo y hardware importado.
  Su capacidad la fija el CapEx (500 MW), su demanda eléctrica es
  inelástica y el hardware tiene precio mundial constante.
* Resto (R): transable (precio mundial = numerario), intensivo en trabajo,
  usa trabajo, capital doméstico y energía (Cobb-Douglas).

Precios endógenos: p_E, w, r_E, r_R (p_R = 1). Dos cierres:

* ``"corto"``: capital específico por sector (r_E != r_R).
* ``"largo"``: capital doméstico móvil entre E y R (r_E = r_R).

Unidades: dinero en miles de millones de USD (USD bn) por año, energía en
TWh, p_E en USD bn/TWh (x1000 = USD/MWh). Trabajo y capital se miden en
unidades de eficiencia con w0 = r0 = 1, de modo que coinciden con su
remuneración en la matriz de contabilidad social (SAM) base.

Al final se corre la simulación completa. Para usar el modelo desde otro notebook: `%run stargate_cge.ipynb`.

In [1]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import root

## Parámetros del shock (datos del proyecto + supuestos)

In [2]:
@dataclass(frozen=True)
class ShockDatacenter:
    """Parámetros empíricos fijos del proyecto (ceteris paribus)."""

    capex_usd_bn: float = 18.9          # stock de capital importado (IED bajo RIGI)
    potencia_mw: float = 500.0          # potencia base
    energia_twh: float = 4.55           # demanda eléctrica anual (inelástica)
    costo_energia_usd_bn: float = 0.305  # factura eléctrica anual a precio base
    # Supuestos (no son datos del proyecto; ajustables):
    masa_salarial_usd_bn: float = 0.06  # ~1.500 puestos directos x USD 40k
    costo_uso_capital: float = 0.25     # r* + delta: 5% + 20% (vida útil hardware ~5 años)
    markup: float = 1.20                # ingreso / costo total a precios base
    alicuota_rigi: float = 0.25         # Ganancias bajo RIGI

    @property
    def precio_energia_base(self) -> float:
        """p_E0 implícito en el costo operativo: 305 / 4,55 = 67 USD/MWh."""
        return self.costo_energia_usd_bn / self.energia_twh

## Economía base (SAM agregada)

In [3]:
@dataclass(frozen=True)
class EconomiaBase:
    """SAM agregada del año base (supuestos de calibración, orden de magnitud Argentina)."""

    pbi: float = 640.0                       # USD bn
    demanda_electrica_twh: float = 140.0     # SADI
    share_electricidad_resto: float = 0.55   # resto = consumo residencial
    share_trabajo_pbi: float = 0.50
    share_trabajo_energia: float = 0.20      # generación: intensiva en capital
    elasticidad_oferta_laboral: float = 0.20  # 0 = oferta de trabajo fija

## Calibración

In [4]:
def calibrar(eco: EconomiaBase, dc: ShockDatacenter) -> dict:
    """Calibración por participaciones: replica la SAM base con w = r = 1."""
    pE0 = dc.precio_energia_base
    XE0 = eco.demanda_electrica_twh
    ER0 = eco.share_electricidad_resto * XE0
    CE0 = XE0 - ER0

    aL = eco.share_trabajo_energia
    LE0 = aL * pE0 * XE0
    KE0 = (1 - aL) * pE0 * XE0

    # PBI = VA_E + VA_R  =>  X_R = PBI - p_E X_E + p_E E_R
    XR0 = eco.pbi - pE0 * XE0 + pE0 * ER0
    L0 = eco.share_trabajo_pbi * eco.pbi
    LR0 = L0 - LE0
    KR0 = XR0 - LR0 - pE0 * ER0
    thL, thK, thE = LR0 / XR0, KR0 / XR0, pE0 * ER0 / XR0

    # Hogar Cobb-Douglas con ocio: elasticidad no compensada de L^s = mu_l * rK / (wL)
    K0 = KE0 + KR0
    mu_l = eco.elasticidad_oferta_laboral * L0 / K0
    F0 = (L0 + K0) / (1 - mu_l)  # ingreso pleno (incluye el valor del ocio)
    T = L0 + mu_l * F0           # dotación de tiempo

    # Datacenter: coeficientes Leontief por MW instalado
    a_ED = dc.energia_twh / dc.potencia_mw
    a_LD = dc.masa_salarial_usd_bn / dc.potencia_mw
    a_KD = dc.capex_usd_bn / dc.potencia_mw
    pD = dc.markup * (pE0 * a_ED + a_LD + dc.costo_uso_capital * a_KD)

    return {
        "pE0": pE0, "XE0": XE0, "XR0": XR0, "KE0": KE0, "KR0": KR0, "T": T,
        "aL": aL, "AE": aL**-aL * (1 - aL) ** -(1 - aL) / pE0,
        "thL": thL, "thK": thK, "thE": thE,
        "AR": thL**-thL * thK**-thK * (pE0 / thE) ** thE,
        "mu_l": mu_l, "mu_E": pE0 * CE0 / F0, "mu_R": XR0 / F0,
        "XD": dc.potencia_mw, "a_ED": a_ED, "a_LD": a_LD, "a_KD": a_KD,
        "pD": pD, "ucap": dc.costo_uso_capital, "tau": dc.alicuota_rigi,
    }

## Comportamiento óptimo de firmas y hogar

In [5]:
def _bloque(c: dict, v: np.ndarray, con_dc: bool) -> dict:
    """Comportamiento óptimo de firmas y hogar dados precios y niveles de actividad."""
    pE, w, rE, rR, XE, XR = v
    aL, thL, thK, thE = c["aL"], c["thL"], c["thK"], c["thE"]

    # Costos unitarios (dual de la maximización de beneficios con CRS)
    cE = (w / aL) ** aL * (rE / (1 - aL)) ** (1 - aL) / c["AE"]
    cR = (w / thL) ** thL * (rR / thK) ** thK * (pE / thE) ** thE / c["AR"]

    # Demandas condicionadas de factores (lema de Shephard)
    LE, KE = aL * cE * XE / w, (1 - aL) * cE * XE / rE
    LR, KR, ER = thL * cR * XR / w, thK * cR * XR / rR, thE * cR * XR / pE

    # Datacenter: capacidad fija, Leontief, hardware a precio mundial
    XD = c["XD"] if con_dc else 0.0
    ED, LD, KD = c["a_ED"] * XD, c["a_LD"] * XD, c["a_KD"] * XD
    PiD = c["pD"] * XD - pE * ED - w * LD - c["ucap"] * KD
    Tax = c["tau"] * max(PiD, 0.0)

    # Hogar: ingreso pleno y demandas Cobb-Douglas
    F = w * c["T"] + rE * c["KE0"] + rR * c["KR0"] + Tax
    ocio = c["mu_l"] * F / w
    CE, CR = c["mu_E"] * F / pE, c["mu_R"] * F

    return dict(pE=pE, w=w, rE=rE, rR=rR, XE=XE, XR=XR, cE=cE, cR=cR,
                LE=LE, KE=KE, LR=LR, KR=KR, ER=ER, XD=XD, ED=ED, LD=LD, KD=KD,
                PiD=PiD, Tax=Tax, F=F, ocio=ocio, Ls=c["T"] - ocio, CE=CE, CR=CR)

## Sistema de ecuaciones de equilibrio

In [6]:
def _residuos(z: np.ndarray, c: dict, cierre: str, con_dc: bool) -> np.ndarray:
    b = _bloque(c, np.exp(z), con_dc)
    res = [
        np.log(b["cE"] / b["pE"]),                          # beneficio cero E
        np.log(b["cR"]),                                     # beneficio cero R (p_R = 1)
        np.log((b["ER"] + b["CE"] + b["ED"]) / b["XE"]),     # vaciado eléctrico
        np.log((b["LE"] + b["LR"] + b["LD"]) / b["Ls"]),     # vaciado laboral
    ]
    if cierre == "corto":
        res += [np.log(b["KE"] / c["KE0"]), np.log(b["KR"] / c["KR0"])]
    elif cierre == "largo":
        res += [np.log((b["KE"] + b["KR"]) / (c["KE0"] + c["KR0"])),
                np.log(b["rE"] / b["rR"])]
    else:
        raise ValueError(f"cierre desconocido: {cierre!r}")
    return np.array(res)

## Solver

In [7]:
def resolver(c: dict, cierre: str = "largo", con_dc: bool = True) -> dict:
    """Resuelve el vector de precios y cantidades de equilibrio (en logs, z > 0)."""
    z0 = np.log([c["pE0"], 1.0, 1.0, 1.0, c["XE0"], c["XR0"]])
    sol = root(_residuos, z0, args=(c, cierre, con_dc), method="hybr", tol=1e-12)
    # hybr puede reportar "sin avances" cuando ya está en la solución: se acepta si el residuo es ~0
    if not sol.success and np.abs(sol.fun).max() > 1e-10:
        raise RuntimeError(f"No convergió ({cierre}, dc={con_dc}): {sol.message}")
    b = _bloque(c, np.exp(sol.x), con_dc)
    b["max_residuo"] = float(np.abs(sol.fun).max())
    return b

## Indicadores y bienestar

In [8]:
def _utilidad(c: dict, b: dict) -> float:
    return b["CR"] ** c["mu_R"] * b["CE"] ** c["mu_E"] * b["ocio"] ** c["mu_l"]

In [9]:
def tabla_resultados(c: dict, escenarios: dict[str, dict]) -> pd.DataFrame:
    """Indicadores por escenario; el primero es la base para Δ% y la VE."""
    base = next(iter(escenarios.values()))
    # Función de gasto Cobb-Douglas a precios base: e(p0, U) = U * prod (p0_i / mu_i)^mu_i
    p0 = {"CR": 1.0, "CE": base["pE"], "ocio": base["w"]}
    mus = {"CR": c["mu_R"], "CE": c["mu_E"], "ocio": c["mu_l"]}
    kappa = np.prod([(p0[k] / m) ** m for k, m in mus.items() if m > 0])
    U0 = _utilidad(c, base)

    filas = {}
    for nombre, b in escenarios.items():
        pib = b["pE"] * b["XE"] + b["XR"] - b["pE"] * b["ER"] + c["pD"] * b["XD"] - b["pE"] * b["ED"]
        absorcion_dc = b["pE"] * b["ED"] + b["w"] * b["LD"] + b["Tax"]
        filas[nombre] = {
            "p_E (USD/MWh)": 1000 * b["pE"],
            "w (salario, base=1)": b["w"],
            "r_E (base=1)": b["rE"],
            "r_R (base=1)": b["rR"],
            "X_E generación (TWh)": b["XE"],
            "  E_R resto (TWh)": b["ER"],
            "  C_E hogares (TWh)": b["CE"],
            "  E_D datacenter (TWh)": b["ED"],
            "X_R resto (USD bn)": b["XR"],
            "L_E": b["LE"], "L_R": b["LR"], "L_D": b["LD"],
            "L oferta": b["Ls"],
            "K_E": b["KE"], "K_R": b["KR"],
            "C_R hogares (USD bn)": b["CR"],
            "Factura eléctrica DC (USD M)": 1000 * b["pE"] * b["ED"],
            "Beneficio DC antes de imp. (USD bn)": b["PiD"],
            "Recaudación RIGI (USD bn)": b["Tax"],
            "PBI (USD bn)": pib,
            "Ingreso nacional hogares (USD bn)": b["F"] - b["w"] * b["ocio"],
            "Déficit comercial en R (USD bn)": b["CR"] - b["XR"],
            "Chequeo Walras (≈0)": (b["CR"] - b["XR"]) - absorcion_dc,
            "Variación equivalente (USD bn)": (_utilidad(c, b) - U0) * kappa,
            "Máx. residuo solver": b["max_residuo"],
        }
    df = pd.DataFrame(filas)
    nombres = list(escenarios)
    for n in nombres[1:]:
        df[f"Δ% {n}"] = np.where(df[nombres[0]].abs() > 1e-9,
                                 100 * (df[n] / df[nombres[0]] - 1), np.nan)
    return df

## Simulación completa

In [10]:
def simular(eco: EconomiaBase | None = None, dc: ShockDatacenter | None = None) -> pd.DataFrame:
    c = calibrar(eco or EconomiaBase(), dc or ShockDatacenter())
    escenarios = {
        "Base": resolver(c, "largo", con_dc=False),
        "Corto plazo": resolver(c, "corto", con_dc=True),
        "Largo plazo": resolver(c, "largo", con_dc=True),
    }
    return tabla_resultados(c, escenarios)

## Resultados

In [11]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
simular()

,Base,Corto plazo,Largo plazo,Δ% Corto plazo,Δ% Largo plazo
p_E (USD/MWh),67.0330,68.8137,67.0369,2.6565,0.0059
"w (salario, base=1)",1.0000,1.0001,0.9999,0.0075,-0.0097
r_E (base=1),1.0000,1.0333,1.0001,3.3296,0.0098
r_R (base=1),1.0000,0.9995,1.0001,-0.0510,0.0098
X_E generación (TWh),140.0000,140.9180,144.5086,0.6557,3.2205
E_R resto (TWh),77.0000,74.9692,76.9435,-2.6374,-0.0734
C_E hogares (TWh),63.0000,61.3988,63.0151,-2.5416,0.0240
E_D datacenter (TWh),0.0000,4.5500,4.5500,NaN,NaN
X_R resto (USD bn),635.7769,635.4528,635.3476,-0.0510,-0.0675
L_E,1.8769,1.9393,1.9377,3.3218,3.2365
